In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, brier_score_loss, classification_report, confusion_matrix
from sklearn.calibration import calibration_curve
import shap
from scipy.stats import randint, uniform

## Import First 24-Hour Dataframe

In [ ]:
df_24hour = pd.read_pickle('micu_24hours.pkl')

# Check shape of dataframe
print("Baseline Data Shape:", df_24hour.shape)

## Split Data into Train, Validation, and Test Sets

In [ ]:
# Separate features and target
X = df_24hour.drop(columns=['stay_id', 'los', 'extended_stay'])
y = df_24hour['extended_stay']

# Save feature names for intepretation
feature_names = X.columns

# Establish test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)

# Establish train and validation sets from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.1111, random_state=42, stratify=y_temp)

## Data Transformation and Preprocessing

In [ ]:
# Convert text columns to Pandas category type
cat_cols = ['gender', 'race', 'admission_type', 'admission_location']

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

## Create & Train LightGBM Model

In [ ]:
lgb_mod = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    importance_type='gain',
    random_state=42,
    verbose=-1
)

lgb_mod.fit(X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            eval_metric='binary_logloss'
           )

## Evalutate Performance on Validation Set

### Generate Validation Predictions

In [ ]:
y_val_pred = lgb_mod.predict(X_val)
y_val_prob = lgb_mod.predict_proba(X_val)[:, 1]

### Validation ROC-AUC and AUPRC

In [ ]:
val_auc = roc_auc_score(y_val, y_val_prob)
val_auprc = average_precision_score(y_val, y_val_prob)

print(f"Validation ROC-AUC: {val_auc:.4f}")
print(f"Validation AUPRC: {val_auprc:.4f}")

#### Given that extended stays make up 25% of the data, **relative improvement from this model's AUPRC score is 87%**.

### Validation ROC Curve

In [ ]:
# Calculate validation false positive rate, true positive rate, and thresholds
fpr_val, tpr_val, thresholds_val = roc_curve(y_val, y_val_prob)

# Plot the validation ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr_val, tpr_val, label=f"Validation ROC (AUC = {val_auc:.4f})")

# Plot the random guess baseline
plt.plot([0, 1], [0, 1], label="Random Guess (AUC = 0.500)")

# Format plot
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation Set Receiver Operating Characteristic (ROC)")
plt.legend(loc='lower right')
plt.show()

### Validation Brier Score

In [ ]:
val_brier = brier_score_loss(y_val, y_val_prob)
print(f"Brier Score Loss: {val_brier:.4f}")

#### Given that extended stays make up 25% of the data, **relative improvement from this model's Brier score is 15%**.

### Validation Calibration Curve (Reliability Diagram)

In [ ]:
# Compute calibration curve data
prob_true_lgb_val, prob_pred_lgb_val = calibration_curve(y_val, y_val_prob, n_bins=10, strategy='quantile')

# Plot curve
plt.figure(figsize=(7, 7))
plt.plot([0,1], [0,1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_lgb_val, prob_true_lgb_val, 's-', label=f"LightGBM (Brier: {val_brier:.3f})")

# Format plot
plt.xlabel("Predicted Probability of Extended Stay")
plt.ylabel("Actual Extended Stay Rate")
plt.title("LightGBM Calibration Curve (Validation Set)")
plt.legend(loc='lower right')
plt.show()

### Validation Feature Importance

In [ ]:
importances = lgb_mod.feature_importances_

# Plot gain importance
lgb.plot_importance(lgb_mod, importance_type='gain', title="LightGMB Feature Importance (Gain)")
plt.show()

### Validation SHAP Values

In [ ]:
explainer = shap.TreeExplainer(lgb_mod)
shap_values = explainer(X_val)
shap.summary_plot(shap_values, X_val)

#### According to this model, high average heart rate, high max temperature, and low blood pressure are the three most influential features in whether a patient stays longer than 4 days in the ICU.

## Hyperparameter Tuning with Randomized Search

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "What parameters are important to include in a randomized search
#### parameter grid for a LightGBM model, and how can I best decide which
#### values to use?"
#### Usage: Included several recommended parameters; reviewed proposed
#### value ranges in conjunction with other resources.
#### -------------------------------------------------------------------------

In [ ]:
# Define hyperparameters to search through
param_grid = {
    'num_leaves': randint(15, 64),
    'min_child_samples': randint(10, 70),
    'max_depth': [-1, 4, 6, 8],
    'learning_rate': uniform(0.01, 0.19),
    'n_estimators': [500],
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': [0.0, 0.1, 1.0, 5.0],
    'reg_lambda': [0.0, 0.1, 1.0, 5.0]
}

# Set up Randomized Search
random_search = RandomizedSearchCV(
    estimator=lgb_mod,
    param_distributions=param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)

# Run search on training data
random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print(f"Best Cross-Validation ROC-AUC: {random_search.best_score_:.4f}")

## Create & Train New LightGBM Model with Optimal Hyperparameters

In [ ]:
lgb_mod_final = lgb.LGBMClassifier(
    **random_search.best_params_,
    random_state=42,
    verbose=-1
)

lgb_mod_final.fit(X_train, y_train)

## Re-evaluate Performance on Validation Set

### Generate New Validation Predictions

In [ ]:
y_val_prob2 = lgb_mod_final.predict_proba(X_val)[:, 1]

### Validation Youden's J Score

In [ ]:
# Calculate new validation false positive rate, true positive rate, and thresholds
fpr_val2, tpr_val2, thresholds_val2 = roc_curve(y_val, y_val_prob2)

val_j = tpr_val2 - fpr_val2

# Find the index of the highest J score
best_idx_val = np.argmax(val_j)
best_threshold_val = thresholds_val2[best_idx_val]
best_j_value_val = val_j[best_idx_val]

print(f"New Validation Optimal J Score Threshold: {best_threshold_val:.4f}")
print(f"New Validation Maximized Youden's J Score: {best_j_value_val:.4f}")

## Evaluate Performance on Test Set

### Generate Test Predictions (Implement Optimal J Score Threshold)

In [ ]:
y_test_prob = lgb_mod_final.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_threshold_val).astype(int)

### Test Metrics

In [ ]:
# Calculate test false positive rate, true positive rate, and thresholds
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_prob)

# Calculate test metrics
test_roc = roc_auc_score(y_test, y_test_prob)
test_auprc = average_precision_score(y_test, y_test_prob)
test_brier = brier_score_loss(y_test, y_test_prob)

print(f"Final Test ROC-AUC: {test_roc:.4f}")
print(f"Final Test AUPRC: {test_auprc:.4f}")
print(f"Final Test Brier Score Loss: {test_brier:.4f}")

print("Final Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=["Standard Stay", "Extended Stay"]))

print("Final Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

#### **Final relative AUPRC improvement over baseline is 106%**.

#### **Final relative Brier score improvement over baseline is 20%**.

### Test Calibration Curve (Reliability Diagram)

In [ ]:
# Compute calibration curve data
prob_true_lgb_test, prob_pred_lgb_test = calibration_curve(y_test, y_test_prob, n_bins=10, strategy='quantile')

# Plot curve
plt.figure(figsize=(7, 7))
plt.plot([0,1], [0,1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_lgb_test, prob_true_lgb_test, 's-', label=f'LightGBM (Brier: {test_brier:.4f})')

plt.xlabel("Predicted Probability of Extended Stay")
plt.ylabel("Actual Extended Stay Rate")
plt.title("LightGBM Calibration Curve (Test Set)")
plt.legend(loc='lower right')
plt.show()